In [ ]:
import os
import csv
from typing import List, Dict
from datetime import datetime
from docx import Document
from docx.shared import Inches, Pt, RGBColor
from docx.enum.text import WD_PARAGRAPH_ALIGNMENT
from docx.oxml import parse_xml
from docx.oxml.ns import nsdecls
from huggingface_hub import InferenceClient

class FreeTechnicalAnalyzer:
    """
    Анализатор ТЗ с использованием критериев из CSV и экспортом в DOCX.
    """
    
    def __init__(self, hf_token: str, model: str = "Qwen/Qwen2.5-72B-Instruct"):
        self.model = model
        self.client = InferenceClient(token=hf_token)
        
    def load_criteria_from_csv(self, csv_path: str) -> List[Dict[str, str]]:
        """
        Загружает критерии из CSV файла.
        Ожидается формат: Название;Описание;Важность
        """
        criteria = []
        try:
            with open(csv_path, mode='r', encoding='utf-8-sig') as file:
                reader = csv.DictReader(file, delimiter=';')
                
                for row in reader:
                    clean_row = {k.strip(): v.strip() for k, v in row.items() if k}
                    if 'Название' in clean_row and 'Описание' in clean_row:
                        criteria.append({
                            'name': clean_row['Название'],
                            'description': clean_row['Описание'],
                            'importance': clean_row.get('Важность', '1')
                        })
            print(f"✅ Загружено {len(criteria)} критериев из {csv_path}")
            return criteria
        except Exception as e:
            print(f"❌ Ошибка чтения CSV: {e}")
            return []

    def extract_text(self, docx_path: str) -> str:
        """Извлекает текст из DOCX"""
        try:
            doc = Document(docx_path)
            full_text = []
            for para in doc.paragraphs:
                if para.text.strip():
                    full_text.append(para.text)
            return '\n'.join(full_text)
        except Exception as e:
            return f"Ошибка чтения файла: {e}"

    def analyze(self, doc_text: str, criteria: List[Dict[str, str]]) -> str:
        """Отправляет текст на анализ в LLM по загруженным критериям"""
        
        if not criteria:
            return "❌ Нет критериев для анализа."

        criteria_str = ""
        for i, c in enumerate(criteria, 1):
            criteria_str += f"{i}. {c['name']} (Важность: {c['importance']})\n   Описание: {c['description']}\n"
        
        prompt = f"""Проанализируй текст технического задания (ТЗ) строго по следующим критериям.
        
СПИСОК КРИТЕРИЕВ:
{criteria_str}

ТЕКСТ ТЗ:
{doc_text[:12000]}

ИНСТРУКЦИЯ:
Для каждого критерия (особенно с Важностью 3) напиши:
1. Статус: [✅ Выполнено / ⚠️ Частично / ❌ Не выполнено]
2. Цитата или обоснование из текста.
3. Если не выполнено — рекомендация, что добавить.

В конце сделай итоговое резюме: готов ли документ к работе.
Отвечай на русском языке.
"""

        messages = [
            {"role": "system", "content": "Ты — строгий технический аудитор. Твоя задача — проверить ТЗ на соответствие списку критериев."},
            {"role": "user", "content": prompt}
        ]

        print(f"⏳ Отправка запроса к модели {self.model}...")
        
        try:
            response_stream = self.client.chat_completion(
                messages=messages,
                model=self.model,
                max_tokens=2500,
                stream=True,
                temperature=0.4
            )
            
            result_text = ""
            print("\n" + "="*40 + "\n ОТЧЕТ ОБ АНАЛИЗЕ \n" + "="*40 + "\n")
            for chunk in response_stream:
                content = chunk.choices[0].delta.content
                if content:
                    print(content, end="", flush=True)
                    result_text += content
            print("\n\n--- КОНЕЦ ОТЧЕТА ---")
            return result_text

        except Exception as e:
            return f"❌ Ошибка API: {e}"

    def _set_cell_background(self, cell, fill_color: str):
        """
        Вспомогательный метод для установки цвета фона ячейки.
        
        Args:
            cell: Ячейка таблицы
            fill_color: HEX код цвета без #, например "003366"
        """
        shd = parse_xml(r'<w:shd {} w:fill="{}"/>'.format(nsdecls('w'), fill_color))
        cell._element.get_or_add_tcPr().append(shd)

    def export_to_docx(self, 
                       analysis_result: str, 
                       criteria: List[Dict[str, str]],
                       source_docx: str,
                       output_path: str = "Analysis_Report.docx") -> bool:
        """
        Экспортирует результаты анализа в DOCX файл с красивым форматированием.
        
        Args:
            analysis_result: Текст результатов анализа от LLM
            criteria: Список использованных критериев
            source_docx: Путь к исходному ТЗ (для справки)
            output_path: Путь к выходному файлу
            
        Returns:
            True если успешно, False если ошибка
        """
        try:
            # Создаём новый документ
            doc = Document()
            
            # Установка стилей и шрифтов
            style = doc.styles['Normal']
            style.font.name = 'Calibri'
            style.font.size = Pt(11)
            
            # ===== ЗАГОЛОВОК =====
            title = doc.add_heading('Отчет об анализе технического задания', 0)
            title.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
            title_format = title.runs[0]
            title_format.font.size = Pt(18)
            title_format.font.bold = True
            title_format.font.color.rgb = RGBColor(0, 51, 102)
            
            # ===== МЕТАИНФОРМАЦИЯ =====
            metadata_heading = doc.add_heading('Информация об анализе', level=1)
            metadata_heading.runs[0].font.color.rgb = RGBColor(0, 51, 102)
            
            meta_table = doc.add_table(rows=4, cols=2)
            meta_table.style = 'Light Grid Accent 1'
            
            # Заполняем таблицу метаданных
            meta_data = [
                ('Дата анализа:', datetime.now().strftime('%d.%m.%Y %H:%M:%S')),
                ('Исходный документ:', os.path.basename(source_docx)),
                ('Модель анализа:', self.model),
                ('Количество критериев:', str(len(criteria)))
            ]
            
            for idx, (label, value) in enumerate(meta_data):
                meta_table.rows[idx].cells[0].text = label
                meta_table.rows[idx].cells[0].paragraphs[0].runs[0].font.bold = True
                meta_table.rows[idx].cells[1].text = value
            
            doc.add_paragraph()  # Пустая строка
            
            # ===== ИСПОЛЬЗУЕМЫЕ КРИТЕРИИ =====
            criteria_heading = doc.add_heading('Использованные критерии', level=1)
            criteria_heading.runs[0].font.color.rgb = RGBColor(0, 51, 102)
            
            criteria_table = doc.add_table(rows=1, cols=4)
            criteria_table.style = 'Light Grid Accent 1'
            
            # Заголовки таблицы критериев
            header_cells = criteria_table.rows[0].cells
            header_cells[0].text = '№'
            header_cells[1].text = 'Критерий'
            header_cells[2].text = 'Описание'
            header_cells[3].text = 'Важность'
            
            # Форматирование заголовков - ИСПРАВЛЕНО
            for cell in header_cells:
                # Устанавливаем цвет фона ячейки
                self._set_cell_background(cell, '003366')
                
                # Форматируем текст в заголовке
                for paragraph in cell.paragraphs:
                    for run in paragraph.runs:
                        run.font.bold = True
                        run.font.color.rgb = RGBColor(255, 255, 255)
            
            # Добавляем критерии в таблицу
            for idx, criterion in enumerate(criteria, 1):
                row_cells = criteria_table.add_row().cells
                row_cells[0].text = str(idx)
                row_cells[1].text = criterion['name']
                row_cells[2].text = criterion['description']
                row_cells[3].text = criterion['importance']
            
            doc.add_paragraph()  # Пустая строка
            
            # ===== РЕЗУЛЬТАТЫ АНАЛИЗА =====
            results_heading = doc.add_heading('Результаты анализа', level=1)
            results_heading.runs[0].font.color.rgb = RGBColor(0, 51, 102)
            
            # Добавляем результаты анализа
            result_para = doc.add_paragraph(analysis_result)
            result_para.paragraph_format.line_spacing = 1.5
            
            # ===== ЗАКЛЮЧЕНИЕ =====
            doc.add_paragraph()
            conclusion_heading = doc.add_heading('Рекомендации', level=1)
            conclusion_heading.runs[0].font.color.rgb = RGBColor(0, 51, 102)
            
            conclusion_text = doc.add_paragraph(
                "1. Внимательно изучите все замечания, отмеченные как ❌ (Не выполнено)\n"
                "2. Приоритизируйте исправления по важности критериев (Важность 3 = критичное)\n"
                "3. После внесения изменений рекомендуется провести повторный анализ\n"
                "4. Убедитесь, что все требования с высокой важностью полностью выполнены"
            )
            conclusion_text.paragraph_format.line_spacing = 1.5
            
            # ===== ПОДВАЛ =====
            doc.add_paragraph()
            footer = doc.add_paragraph("Отчет сгенерирован автоматически системой анализа ТЗ")
            footer.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
            footer_run = footer.runs[0]
            footer_run.font.size = Pt(9)
            footer_run.font.italic = True
            footer_run.font.color.rgb = RGBColor(128, 128, 128)
            
            # Сохраняем документ
            doc.save(output_path)
            print(f"✅ Отчет успешно сохранен: {output_path}")
            return True
            
        except Exception as e:
            print(f"❌ Ошибка при экспорте в DOCX: {e}")
            import traceback
            traceback.print_exc()
            return False


# ==========================================
# ИНСТРУКЦИЯ ПО ЗАПУСКУ
# ==========================================

# 1. Получите токен: https://huggingface.co/settings/tokens
HF_TOKEN = "" 

# 2. Инициализация
analyzer = FreeTechnicalAnalyzer(hf_token=HF_TOKEN)

# 3. Пути к файлам
csv_file = r'C:\Users\troyd\OneDrive\Desktop\Стажировка\Критерии.csv'
docx_file = r'C:\Users\troyd\OneDrive\Desktop\Стажировка\ТЗ Электронный журнал_v2.1.docx'
output_report = "Analysis_Report.docx"

# 4. Основной процесс
if os.path.exists(csv_file) and os.path.exists(docx_file):
    # Шаг A: Читаем критерии
    loaded_criteria = analyzer.load_criteria_from_csv(csv_file)
    
    # Шаг B: Читаем документ
    doc_text = analyzer.extract_text(docx_file)
    
    # Шаг C: Анализируем
    if loaded_criteria and doc_text:
        analysis_result = analyzer.analyze(doc_text, loaded_criteria)
        
        # Шаг D: Экспортируем результаты в DOCX
        analyzer.export_to_docx(
            analysis_result=analysis_result,
            criteria=loaded_criteria,
            source_docx=docx_file,
            output_path=output_report
        )
else:
    print(f"⚠️ Проверьте наличие файлов:\n- {csv_file}: {os.path.exists(csv_file)}\n- {docx_file}: {os.path.exists(docx_file)}")


✅ Загружено 26 критериев из C:\Users\troyd\OneDrive\Desktop\Стажировка\Критерии.csv
⏳ Отправка запроса к модели Qwen/Qwen2.5-72B-Instruct...

 ОТЧЕТ ОБ АНАЛИЗЕ 

### Анализ Технического Задания

#### 1. Полнота функциональных требований (Важность: 3)
**Статус:** ✅ Выполнено
**Цитата или обоснование:** В разделе 5.3 подробно описаны все журналы, их структура и требования к заполнению. В разделе 5.4 описаны требования к программному обеспечению, включая авторизацию, единую точку входа и ввод данных.
**Рекомендация:** Нет.

#### 2. Детализация объектов (Важность: 3)
**Статус:** ✅ Выполнено
**Цитата или обоснование:** В разделе 5.3 подробно описаны все журналы, их атрибуты и типы данных. В приложениях 2-27 приведены формы журналов с детализацией полей.
**Рекомендация:** Нет.

#### 3. Бизнес-логика (Важность: 3)
**Статус:** ✅ Выполнено
**Цитата или обоснование:** В разделе 5.3 описаны правила заполнения журналов, условия переходов между состояниями записей, валидация ввода данных и другие п